In [70]:
import re
import spacy

import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import NMF
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

In [71]:
# Load the data
file_name = '../data/ncsl_ai_db_clean.csv'
df = pd.read_csv(file_name)

# Fill in empty strings for null text columns
df["title"] = df.title.fillna("")
df["summary"] = df.summary.fillna("")

In [72]:
# Map the bill status sub-types to main types - {Passed, Failed, Pending, Vetoed}
bill_status_map = {
    "Enacted": "Passed",
    "Failed": "Failed",
    "Failed - Adjourned": "Failed",
    "Adopted": "Passed",
    "Pending": "Pending",
    "Pending - Carryover": "Pending",
    "To Governor": "Pending",
    "Override Pending": "Pending",
    "Vetoed": "Vetoed"
}
df["status_type"] = df.status.map(bill_status_map)

unmapped_status = df.loc[df.status_type.isna(), "status"].unique()
assert len(
    unmapped_status) == 0, f"Statuses missing from bill_status_map: {unmapped_status}"

# Bill outcome flags
df["outcome_final"] = df.status_type != "Pending"
df["status_stale"] = (df.status_type == "Pending") & (
    df.page_year < df.page_year.max())

In [73]:
# Map the audience from the bill_number to the session's chamber
bill_audience_map = {'A': 'Assembly Bill',
                     'ACR': 'Assembly Concurrent Resolution',
                     'AJR': 'Assembly Joint Resolution',
                     'AR': 'Assembly Resolution',
                     'B': 'Bill',
                     'BDR': 'Bill Draft Request',
                     'D': 'Draft Bill',
                     'H': 'House Bill',
                     'HCR': 'House Concurrent Resolution',
                     'HD': 'House Docket',
                     'HJM': 'House Joint Memorial',
                     'HJR': 'House Joint Resolution',
                     'HM': 'House Memorial',
                     'HR': 'House Resolution',
                     'HSB': 'House Study Bill',
                     'L': 'Legislative Bill',
                     'LCR': 'Legislative Council Resolution',
                     'LR': 'Legislative Reference',
                     'S': 'Senate Bill',
                     'SCR': 'Senate Concurrent Resolution',
                     'SD': 'Senate Docket',
                     'SJR': 'Senate Joint Resolution',
                     'SM': 'Senate Memorial',
                     'SPB': 'Senate Bill',
                     'SR': 'Senate Resolution',
                     'SSB': 'Senate Study Bill',
                     'WS': 'Senate Bill'}

# Split 'bill_number' into chamber prefix and bill number into columns
prefix_number = df.bill_number.str.extract(r"^([A-Za-z]+)\s*(\d+)")
unparsed = df.loc[prefix_number.isna().any(axis=1), "bill_number"].unique()
assert len(
    unparsed) == 0, f"Bill numbers without a prefix and number: {unparsed}"

# Get full chamber name from the prefix
df["chamber_prefix"] = prefix_number[0].str.upper()
df["chamber_name"] = df.chamber_prefix.map(bill_audience_map)
unmapped_prefix = df.loc[df.chamber_name.isna(), "chamber_prefix"].unique()
assert len(
    unmapped_prefix) == 0, f"Prefixes missing from bill_audience_map: {unmapped_prefix}"

# Fix Nebraska LR bills - special case
is_ne_lr = (df.state == "NE") & (df.chamber_prefix == "LR")
df.loc[is_ne_lr, "chamber_name"] = "Legislative Resolution"

# Clean bill number and create an integer type column for order bill introduced
df["order_introduced"] = prefix_number[1].astype(int)
df["bill_number"] = df.bill_number.str.replace(" ", "")

In [74]:
# Chamber (from the bill prefix). "Other" = drafts, legislative references, DC/territory bills
df["chamber"] = df.chamber_prefix.str[0].map(
    {"H": "House", "S": "Senate", "A": "Assembly"}).fillna("Other")
df.loc[df.chamber_prefix == "WS", "chamber"] = "Senate"
df["is_lower_chamber"] = df.chamber.isin(["House", "Assembly"])

# Correct resolutions typed as bills (RI/SC use bill-style prefixes; their summary starts with a "(Resolution)" tag)
res_tag = r"^\((?:Senate |House )?Resolution\)\s*"
is_tagged = df.summary.str.contains(res_tag, regex=True)
df.loc[is_tagged, "chamber_name"] = df.loc[is_tagged,
                                           "chamber_name"].str.replace("Bill", "Resolution")
df["summary"] = df.summary.str.replace(res_tag, "", regex=True)

In [75]:
document_types = ["Resolution", "Study Bill",
                  "Memorial", "Docket", "Draft", "Reference"]

pattern = f"(?i)({'|'.join(document_types)})"

# Extract matching document type, with "Bill" as the default (now that bill-types are fixed)
df["document_type"] = df.chamber_name.str.extract(
    pattern, expand=False).fillna("Bill")

In [76]:
# Encode different types of legislation
df["is_resolution"] = df.chamber_name.str.contains("Resolution")
df["is_joint_resolution"] = df.chamber_name.str.contains("Joint Resolution")
df["is_concurrent_resolution"] = df.chamber_name.str.contains(
    "Concurrent Resolution")
df["is_simple_resolution"] = df.chamber_prefix.isin(["HR", "SR", "AR"]) | is_ne_lr

In [77]:
# Gets the type of each state/area - {State, Territory, District}
df['jurisdiction_type'] = df.state.map(
    lambda s: "Territory" if s in {"PR", "VI", "GU", "AS", "MP"}
    else "District" if s == "DC" else "State")

region_map = {
    **dict.fromkeys(["CT", "ME", "MA", "NH", "RI", "VT", "NJ", "NY", "PA"], "Northeast"),
    **dict.fromkeys(["IL", "IN", "MI", "OH", "WI", "IA", "KS", "MN", "MO", "NE", "ND", "SD"], "Midwest"),
    **dict.fromkeys(["DE", "DC", "FL", "GA", "MD", "NC", "SC", "VA", "WV", "AL", "KY", "MS", "TN",
                     "AR", "LA", "OK", "TX"], "South"),
    **dict.fromkeys(["AZ", "CO", "ID", "MT", "NV", "NM", "UT", "WY", "AK", "CA", "HI", "OR", "WA"], "West"),
    **dict.fromkeys(["PR", "VI", "GU", "AS", "MP"], "Territory"),
}

df["census_region"] = df.state.map(region_map)
assert df.census_region.notna().all(
), df.loc[df.census_region.isna(), "state"].unique()

In [78]:
def any_of(names):
    """1 if the bill has any of these categories (a category no bill has counts as 0)"""
    cols = [f"category_{n}" for n in names]
    return df.reindex(columns=cols, fill_value=0).max(axis=1)


# One hot encoding categories from multi-label text
# Split strings by separator, strip whitespace, and create dummy variables
df.category = df.category.fillna("").str.lower()
dummies = df.category.str.get_dummies(sep=";")
dummies = dummies.rename(columns=lambda c: "category_" +
                         re.sub(r"\W+", "_", c).strip("_"))
# blank label from uncategorized bills, not a category
dummies = dummies.drop(columns="category_", errors="ignore")

In [79]:
# Total number of categories
df["num_categories"] = dummies.sum(axis=1)
df["is_uncategorized"] = df.num_categories == 0

# Add new category columns to existing dataset
df = pd.concat([df, dummies], axis=1)

# Whether a bill was for study-use only
df["studies_only"] = (any_of(["studies"]) == 1) & (df.num_categories == 1)

In [80]:
# Category groups (from the one-hot flags above)
df["catgroup_public_sector"] = any_of(
    ["government_use", "judicial_use", "appropriations", "audit"])
df["catgroup_regulatory"] = any_of(["private_sector", "notification", "private_right_of_action",
                                    "impact_assessment", "oversight_governance", "responsible_use"])
df["catgroup_rights_safety"] = any_of(
    ["criminal_use", "child_pornography", "elections", "personhood", "category_provenance"])
df["catgroup_sector_specific"] = any_of(["health_use", "education_use", "education_training", "housing",
                                         "energy", "effect_on_labor_employment", "cybersecurity", "taxes"])

# 'Education Use' only exists on the 2024+ pages, so catgroup_education works across years
df["catgroup_education"] = any_of(["education_use", "education_training"])

In [81]:
# Basic text structure features
df["title_missing"] = df.title.eq("")
df["title_word_count"] = df.title.str.split().str.len().fillna(0).astype(int)

df["summary_word_count"] = df.summary.str.split().str.len().astype(int)
df["summary_char_count"] = df.summary.str.len()
df["summary_clause_count"] = df.summary.str.count(";") + 1
df["summary_is_vague"] = df.summary_word_count <= 8

In [82]:
nlp = spacy.load("en_core_web_sm")

verbs_list, verb_counts = [], []
for summary in df["summary"]:
    # Skip empty summaries
    if pd.isna(summary):
        verbs_list.append("")
        verb_counts.append(0)
        continue

    doc = nlp(str(summary))

    # Keep words that spaCy tags as verbs
    verbs = []
    for token in doc:
        if token.pos_ == "VERB":
            verbs.append(token.text)

    verbs_list.append(", ".join(verbs))
    verb_counts.append(len(verbs))

df["summary_verbs"] = verbs_list
df["summary_verb_count"] = verb_counts

In [83]:
# Topic modeling 'title' and 'summary'
df["text"] = (df.title + " " + df.summary).str.lower()

# TF-IDF, with extra stop words related to this topic
extra_stop = ["ai", "artificial", "intelligence", "relates", "relating", "provides",
              "certain", "act", "requires", "establishes", "use", "state", "bill",
              "makes", "concerning", "regarding"]
vectorizer = TfidfVectorizer(stop_words=list(ENGLISH_STOP_WORDS) + extra_stop,
                             ngram_range=(1, 2), min_df=5, max_df=0.5)
tfidf = vectorizer.fit_transform(df["text"])
terms = vectorizer.get_feature_names_out()

# Find 25 topics and give each bill its main topic
nmf = NMF(n_components=25, random_state=0, max_iter=500)
bill_topics = nmf.fit_transform(tfidf)
df["main_topic"] = bill_topics.argmax(axis=1)

# Category labels are predictors
# View results, cross-validate them, and identify novel topics
# print("AUC   Bills  Topic")
# X = df[[c for c in df.columns if c.startswith("category_")]]

# for t in range(25):
#     top_words = [terms[j] for j in nmf.components_[t].argsort()[::-1][:5]]
#     y = df["main_topic"] == t
#     auc = cross_val_score(LogisticRegression(max_iter=1000),
#                           X, y, cv=5, scoring="roc_auc").mean()

    # print(f"{auc:.2f}  {y.sum():5}  {', '.join(top_words)}")

In [84]:
# Using topics with lowest AUC-score after viewing content of their bills
topics = {
    "topic_likeness_rights": [r"\blikeness", r"digital replica", r"right of publicity", r"\bvoice and\b"],
    "topic_consumer_protection": [r"consumer protection", r"unfair or deceptive", r"deceptive trade",
                                  r"unfair trade", r"deceptive act"],
    "topic_high_risk_ai": [r"high-risk", r"high risk", r"consequential decision", r"algorithmic discrimination"],
    "topic_data_privacy": [r"privacy", r"personal data", r"personal information", r"biometric"],
}

# Tag the topics using keywords just identified
for column, keywords in topics.items():
    pattern = "|".join(keywords)
    df[column] = df.text.str.contains(pattern, regex=True).astype(int)

In [85]:
history = pd.read_csv("../data/bill_status_history.csv")
history = history.sort_values(["bill_id", "page_year"])

# Mark rows where the status is different from the bill's previous listing
previous_status = history.groupby("bill_id").status.shift()
history["status_changed"] = previous_status.notna() & (history.status != previous_status)

# Summarize each bill's history
bill_histories = history.groupby("bill_id")

history_features = pd.DataFrame({
    "num_listings": bill_histories.page_year.nunique(),
    "first_page_year": bill_histories.page_year.min(),
    "last_page_year": bill_histories.page_year.max(),
    "first_status": bill_histories.status.first(),
    "num_status_changes": bill_histories.status_changed.sum(),
})

history_features["is_carryover"] = history_features.num_listings > 1
history_features["status_changed"] = history_features.num_status_changes > 0
history_features["first_status_type"] = history_features.first_status.map(bill_status_map)

# Add the new columns to the bills dataset
df = df.merge(history_features, left_on="bill_id", right_index=True, how="left")

In [86]:
# Final data quality checks
assert df.bill_id.is_unique
assert (df.session_year <= df.page_year).all()
assert df.state.notna().all() and df.bill_number.notna().all()

df = df.drop(columns=["category", "chamber_name", "page_year",
                      "text", "summary", "main_topic"])

missing = df.isnull().sum()[lambda s: s > 0]
assert missing.empty, f"Missing values:\n{missing}"

# Export final csv to dataset
file_name = "../data/ncsl_ai_db_final.csv"
df.to_csv(file_name, index=False)

# View final columns for  bills dataset
print(df.columns)

Index(['bill_id', 'bill_number', 'state', 'session_year', 'status', 'title',
       'is_special_session', 'status_type', 'outcome_final', 'status_stale',
       'chamber_prefix', 'order_introduced', 'chamber', 'is_lower_chamber',
       'document_type', 'is_resolution', 'is_joint_resolution',
       'is_concurrent_resolution', 'is_simple_resolution', 'jurisdiction_type',
       'census_region', 'num_categories', 'is_uncategorized',
       'category_appropriations', 'category_audit',
       'category_child_pornography', 'category_criminal_use',
       'category_cybersecurity', 'category_education_use',
       'category_education_training', 'category_effect_on_labor_employment',
       'category_elections', 'category_energy', 'category_government_use',
       'category_health_use', 'category_housing', 'category_impact_assessment',
       'category_judicial_use', 'category_notification',
       'category_oversight_governance', 'category_personhood',
       'category_private_right_of_actio